In [ ]:
import pandas as pd
import numpy as np
import nltk
import torch
from transformers import AutoTokenizer, AutoModel

nltk.download('punkt')        # ---------> 'punkt' is a pre-trained NLTK tokenizer model that enables correct splitting of text into words or sentences.
nltk.download('stopwords')    # ---------> 'stopwords' is an NLTK resource that provides lists of common words (like “the”, “and”) to filter out during text preprocessing.
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv("/content/processed_data.csv")
data.head()

,QuestionText,Category,Answer,Question_for_classification,Answer_for_classification,Question_for_QA,Answer_for_QA,Question_for_translation,Answer_for_translation,Question_tokens,Answer_tokens,Question_tokens_no_stopwords,Answer_tokens_no_stopwords,Question_tokens_stemmed,Answer_tokens_stemmed,Question_text_no_stopwords,Question_text_stemmed,Answer_text_no_stopwords,Answer_text_stemmed
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,"['ايهما', 'افضل', 'الدراسة', 'في', 'السابق', '...","['الدراسة', 'في', 'الوقت', 'الحالي', 'تعتبر', ...","['ايهما', 'افضل', 'الدراسة', 'السابق', 'ام', '...","['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'افضل'...","['ايه', 'فضل', 'درس', 'سبق', 'ام', 'وقت', 'الح...","['درس', 'وقت', 'الحالي', 'عبر', 'فضل', 'سبب', ...",ايهما افضل الدراسة السابق ام الوقت الحالي,ايه فضل درس سبق ام وقت الحالي,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...,درس وقت الحالي عبر فضل سبب وفر كنولوج ورد علم حدث
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروة في مصر,القطن يعتبر من اهم المنتجات الزراعية في مصر وي...,اليس القطن عماد الثروة في مصر؟,القطن يعتبر من اهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروة في مصر؟,القطن يعتبر من اهم المنتجات الزراعية في مصر، و...,"['اليس', 'القطن', 'عماد', 'الثروة', 'في', 'مصر']","['القطن', 'يعتبر', 'من', 'اهم', 'المنتجات', 'ا...","['اليس', 'القطن', 'عماد', 'الثروة', 'مصر']","['القطن', 'يعتبر', 'من', 'اهم', 'المنتجات', 'ا...","['الس', 'قطن', 'عمد', 'ثرة', 'مصر']","['قطن', 'عبر', 'من', 'اهم', 'نتج', 'زرع', 'مصر...",اليس القطن عماد الثروة مصر,الس قطن عمد ثرة مصر,القطن يعتبر من اهم المنتجات الزراعية مصر ويعد ...,قطن عبر من اهم نتج زرع مصر يعد من عمد ريس قصد صري
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق,الشمس تصعد من الشرق,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,"['اتصعد', 'الشمس', 'من', 'الشرق']","['الشمس', 'تصعد', 'من', 'الشرق']","['اتصعد', 'الشمس', 'من', 'الشرق']","['الشمس', 'تصعد', 'من', 'الشرق']","['صعد', 'شمس', 'من', 'شرق']","['شمس', 'صعد', 'من', 'شرق']",اتصعد الشمس من الشرق,صعد شمس من شرق,الشمس تصعد من الشرق,شمس صعد من شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حية دقيقة,البكتيريا تعرف بانها كاينات حية دقيقة,اتعرف البكتيريا بانها كاينات حية دقيقة؟,البكتيريا تعرف بانها كاينات حية دقيقة.,اتعرف البكتيريا بانها كاينات حية دقيقة؟,البكتيريا تعرف بانها كاينات حية دقيقة.,"['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حية...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حية'...","['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حية...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حية'...","['عرف', 'كتر', 'بان', 'كين', 'حية', 'دقق']","['كتر', 'عرف', 'بان', 'كين', 'حية', 'دقق']",اتعرف البكتيريا بانها كاينات حية دقيقة,عرف كتر بان كين حية دقق,البكتيريا تعرف بانها كاينات حية دقيقة,كتر عرف بان كين حية دقق
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهوا اساسا من النيتروجين,الهوا يتكون اساسا من النيتروجين,ايتكون الهوا اساسا من النيتروجين؟,الهوا يتكون اساسا من النيتروجين.,ايتكون الهوا اساسا من النيتروجين؟,الهوا يتكون اساسا من النيتروجين.,"['ايتكون', 'الهوا', 'اساسا', 'من', 'النيتروجين']","['الهوا', 'يتكون', 'اساسا', 'من', 'النيتروجين']","['ايتكون', 'الهوا', 'اساسا', 'من', 'النيتروجين']","['الهوا', 'يتكون', 'اساسا', 'من', 'النيتروجين']","['ايت', 'هوا', 'سسا', 'من', 'ترج']","['هوا', 'يتك', 'سسا', 'من', 'ترج']",ايتكون الهوا اساسا من النيتروجين,ايت هوا سسا من ترج,الهوا يتكون اساسا من النيتروجين,هوا يتك سسا من ترج


In [ ]:
questions = data["Question_for_QA"].astype(str).tolist()
answers = data["Answer_for_QA"].astype(str).tolist()

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained('aubmindlab/bert-base-arabertv02')
bert_model = AutoModel.from_pretrained('aubmindlab/bert-base-arabertv02')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_bert_token_embeddings(texts, batch_size=16, max_len=64):
    bert_model.eval()
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        inputs = bert_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_len
        )

        with torch.no_grad():
            outputs = bert_model(**inputs)

        token_embeddings = outputs.last_hidden_state
        all_embeddings.append(token_embeddings.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)

In [ ]:
X_bert_questions = get_bert_token_embeddings(questions,batch_size=16,max_len=64)
X_bert_questions.shape

(4821, 64, 768)

In [ ]:
np.save("bert_QA_question_token_embeddings.npy", X_bert_questions)

In [ ]:
X_bert_answers = get_bert_token_embeddings(answers,batch_size=16,max_len=64)
X_bert_answers.shape

(4821, 64, 768)

In [ ]:
np.save("bert_QA_answer_token_embeddings.npy", X_bert_answers)

In [ ]:
import ast
data['Question_tokens'] = data['Question_tokens'].apply(ast.literal_eval)

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653909 sha256=a02060fe29ea3d9a6eb87f2c3ce871eadab6d5f4866b37660fe0f6c4dd525db0
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
import fasttext

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz

--2026-06-07 14:23:52--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.62, 13.249.182.81, 13.249.182.39, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4500982519 (4.2G) [application/octet-stream]
Saving to: ‘cc.ar.300.bin.gz’

cc.ar.300.bin.gz    100%[===================>]   4.19G   182MB/s    in 32s     

2026-06-07 14:24:24 (133 MB/s) - ‘cc.ar.300.bin.gz’ saved [4500982519/4500982519]



In [ ]:
!gunzip cc.ar.300.bin.gz

In [ ]:
ft_model = fasttext.load_model("/content/cc.ar.300.bin")

In [ ]:
def get_fasttext_sequence(words, max_len=64):
    vectors = []

    for word in words[:max_len]:
        vectors.append(ft_model.get_word_vector(word))

    while len(vectors) < max_len:
        vectors.append(np.zeros(300))

    return np.array(vectors)

In [ ]:
X_fasttext_qa = np.array(data["Question_tokens"].apply(lambda x: get_fasttext_sequence(x)).tolist())
X_fasttext_qa.shape

(4821, 64, 300)

In [ ]:
np.save("fasttext_QA_token_embeddings.npy", X_fasttext_qa)

In [ ]:
import ast
data["Answer_tokens"] = data["Answer_tokens"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
X_fasttext_answers = np.array(data["Answer_tokens"].apply(lambda x: get_fasttext_sequence(x)).tolist())
X_fasttext_answers.shape

(4821, 64, 300)

In [ ]:
np.save("fasttext_answer_token_embeddings.npy", X_fasttext_answers)

In [ ]:
model_name = "Qwen/Qwen3-Embedding-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
model = AutoModel.from_pretrained(model_name,trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
qwen_model = model.to(device)
qwen_model.eval()

def get_qwen_token_embeddings(texts, batch_size=8, max_len=64):
    all_embeddings = []
    texts = list(texts)

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_len
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = qwen_model(**inputs)

        token_embeddings = outputs.last_hidden_state.float().cpu().numpy()
        all_embeddings.append(token_embeddings)


    return np.concatenate(all_embeddings, axis=0)

In [ ]:
X_qwen_qa = get_qwen_token_embeddings(data["Question_for_QA"],batch_size=8,max_len=64)
X_qwen_qa.shape

(4821, 64, 1024)

In [ ]:
np.save("/content/drive/MyDrive/NLP_embedding/qwen_QA_token_embeddings.npy",X_qwen_qa)

In [ ]:
X_qwen_answers = get_qwen_token_embeddings(data["Answer_for_QA"],batch_size=8,max_len=64)
X_qwen_answers.shape

(4821, 64, 1024)

In [ ]:
np.save("/content/drive/MyDrive/NLP_embedding/qwen_answer_token_embeddings.npy",X_qwen_answers)

In [ ]:
model_name = "intfloat/multilingual-e5-large"
e5_tokenizer = AutoTokenizer.from_pretrained(model_name)
e5_model  = AutoModel.from_pretrained(model_name)

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
e5_model = e5_model.to(device)
e5_model.eval()

def get_e5_token_embeddings(texts, batch_size=8, max_len=64):
    all_embeddings = []
    texts = list(texts)

    # E5 prefers prefix
    texts = [f"passage: {t}" for t in texts]

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        inputs = e5_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_len
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = e5_model(**inputs)

        token_embeddings = outputs.last_hidden_state.float().cpu().numpy()
        all_embeddings.append(token_embeddings)

    return np.concatenate(all_embeddings, axis=0)

In [ ]:
X_e5_qa = get_e5_token_embeddings(data["Question_for_QA"],batch_size=8,max_len=64)
print(X_e5_qa.shape)

(4821, 64, 1024)


In [ ]:
np.save("/content/drive/MyDrive/NLP_embedding/e5_QA_token_embeddings.npy",X_e5_qa)

In [ ]:
X_e5_answers = get_e5_token_embeddings(data["Answer_for_QA"],batch_size=8,max_len=64)
print(X_e5_answers.shape)

(4821, 64, 1024)


In [ ]:
np.save("/content/drive/MyDrive/NLP_embedding/e5_answer_token_embeddings.npy",X_e5_answers)

In [ ]:
model_name = "BAAI/bge-m3"
bge_tokenizer = AutoTokenizer.from_pretrained(model_name)
bge_model  = AutoModel.from_pretrained(model_name)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
bge_model = bge_model.to(device)
bge_model.eval()

def get_bge_token_embeddings(texts, batch_size=8, max_len=64):
    all_embeddings = []
    texts = list(texts)

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        inputs = bge_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_len
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = bge_model(**inputs)

        token_embeddings = outputs.last_hidden_state.float().cpu().numpy()
        all_embeddings.append(token_embeddings)

    return np.concatenate(all_embeddings, axis=0)

In [ ]:
X_bge_qa = get_bge_token_embeddings(data["Question_for_QA"],batch_size=8,max_len=64)
print(X_bge_qa.shape)

(4821, 64, 1024)


In [ ]:
np.save("/content/drive/MyDrive/NLP_embedding/bge_QA_token_embeddings.npy",X_bge_qa)

In [ ]:
X_bge_answers = get_bge_token_embeddings(data["Answer_for_QA"],batch_size=8,max_len=64)
print(X_bge_answers.shape)

(4821, 64, 1024)


In [ ]:
np.save("/content/drive/MyDrive/NLP_embedding/bge_answer_token_embeddings.npy",X_bge_answers)

# Seq to Seq

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import gc
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, SimpleRNN,LSTM, GRU, Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

In [ ]:
MAX_LEN = 64
DEC_LEN = 63

In [ ]:
questions = data['Question_for_QA'].astype(str).tolist()
answers = data['Answer_for_QA'].astype(str).tolist()

target_texts = ['starttoken ' + ans + ' endtoken' for ans in answers]

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)

target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences,maxlen=MAX_LEN,padding="post",truncating="post")

In [ ]:
decoder_target_tokens = target_padded[:, 1:]

decoder_target_tokens = decoder_target_tokens.reshape(
    decoder_target_tokens.shape[0],
    decoder_target_tokens.shape[1],
    1
)

decoder_vocab_size = len(target_tokenizer.word_index) + 1

print("Decoder target:", decoder_target_tokens.shape)
print("Decoder vocab:", decoder_vocab_size)

Decoder target: (4821, 63, 1)
Decoder vocab: 15974


In [ ]:
def build_lstm_seq2seq(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=64):
    encoder_inputs = Input(shape=(MAX_LEN, encoder_dim))
    _, state_h, state_c = LSTM(hidden_units, return_state=True)(encoder_inputs)

    decoder_inputs = Input(shape=(DEC_LEN, decoder_dim))
    decoder_outputs, _, _ = LSTM(
        hidden_units,
        return_sequences=True,
        return_state=True
    )(decoder_inputs, initial_state=[state_h, state_c])

    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
def build_gru_seq2seq(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=64):
    encoder_inputs = Input(shape=(MAX_LEN, encoder_dim))
    _, state_h = GRU(hidden_units, return_state=True)(encoder_inputs)

    decoder_inputs = Input(shape=(DEC_LEN, decoder_dim))
    decoder_outputs, _ = GRU(
        hidden_units,
        return_sequences=True,
        return_state=True
    )(decoder_inputs, initial_state=state_h)

    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

In [ ]:
def build_rnn_seq2seq(encoder_dim, decoder_dim, decoder_vocab_size, hidden_units=64):
    encoder_inputs = Input(shape=(MAX_LEN, encoder_dim))
    _, state_h = SimpleRNN(hidden_units, return_state=True)(encoder_inputs)

    decoder_inputs = Input(shape=(DEC_LEN, decoder_dim))
    decoder_outputs, _ = SimpleRNN(
        hidden_units,
        return_sequences=True,
        return_state=True
    )(decoder_inputs, initial_state=state_h)

    decoder_outputs = Dense(decoder_vocab_size, activation="softmax")(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
def train_all_models_for_one_embedding(embedding_name, q_file, a_file, epochs=5, batch_size=16):

    X_q = np.load(q_file)
    X_a = np.load(a_file)

    decoder_input_emb = X_a[:, :-1, :]

    X_train, X_test, dec_in_train, dec_in_test, dec_tar_train, dec_tar_test = train_test_split(
        X_q,
        decoder_input_emb,
        decoder_target_tokens,
        test_size=0.2,
        random_state=42
    )

    builders = {
        "LSTM": build_lstm_seq2seq,
        "GRU": build_gru_seq2seq,
        "SimpleRNN": build_rnn_seq2seq
    }

    results = {}

    for model_name, builder in builders.items():

        print("\n" + "="*60)
        print(f"Training {model_name} with {embedding_name}")
        print("="*60)

        tf.keras.backend.clear_session()
        gc.collect()

        model = builder(
            encoder_dim=X_q.shape[2],
            decoder_dim=decoder_input_emb.shape[2],
            decoder_vocab_size=decoder_vocab_size,
            hidden_units=128
        )

        history = model.fit(
            [X_train, dec_in_train],
            dec_tar_train,
            batch_size=batch_size,
            epochs=epochs,
            validation_data=([X_test, dec_in_test], dec_tar_test),
            verbose=1
        )

        results[model_name] = {
            "model": model,
            "history": history.history,
            "X_test": X_test,
            "dec_in_test": dec_in_test,
            "dec_tar_test": dec_tar_test,
            "final_val_loss": history.history["val_loss"][-1],
            "final_val_accuracy": history.history["val_accuracy"][-1]
        }

    return results

In [ ]:
def summarize_embedding_results(embedding_name, results):
    rows = []

    for model_name, res in results.items():
        rows.append({
            "Embedding": embedding_name,
            "Model": model_name,
            "Validation Accuracy": res["final_val_accuracy"],
            "Validation Loss": res["final_val_loss"]
        })

    return pd.DataFrame(rows)

In [ ]:
fasttext_results = train_all_models_for_one_embedding(
    "FastText",
    "/content/drive/MyDrive/NLP_embedding/fasttext_QA_token_embeddings.npy",
    "/content/drive/MyDrive/NLP_embedding/fasttext_answer_token_embeddings.npy",
    epochs=5,
    batch_size=16
)


Training LSTM with FastText
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - accuracy: 0.7153 - loss: 3.2958 - val_accuracy: 0.7216 - val_loss: 2.3373
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - accuracy: 0.7318 - loss: 2.2099 - val_accuracy: 0.7384 - val_loss: 2.2074
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - accuracy: 0.7488 - loss: 2.0730 - val_accuracy: 0.7582 - val_loss: 2.1062
Epoch 4/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - accuracy: 0.7651 - loss: 1.9539 - val_accuracy: 0.7673 - val_loss: 2.0131
Epoch 5/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - accuracy: 0.7726 - loss: 1.8330 - val_accuracy: 0.7763 - val_loss: 1.9118

Training GRU with FastText
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step - accuracy: 0.7191 - loss: 3.3292 - val_accuracy: 0.7213 - val_loss: 2.4427
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 53ms/step - accuracy: 0.7374 - loss: 2.2355 - val_accuracy: 0.7538 - val_loss: 2.1558
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
summarize_embedding_results("FastText", fasttext_results)

,Embedding,Model,Validation Accuracy,Validation Loss
0,FastText,LSTM,0.776297,1.911823
1,FastText,GRU,0.805790,1.652239
2,FastText,SimpleRNN,0.754684,2.196519


In [ ]:
bert_results = train_all_models_for_one_embedding(
    "BERT",
    "/content/drive/MyDrive/NLP_embedding/bert_QA_question_token_embeddings.npy",
    "/content/drive/MyDrive/NLP_embedding/bert_QA_answer_token_embeddings.npy",
    epochs=5,
    batch_size=16
)


Training LSTM with BERT
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 20s 60ms/step - accuracy: 0.7169 - loss: 3.0986 - val_accuracy: 0.7245 - val_loss: 2.3460
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - accuracy: 0.7273 - loss: 2.2119 - val_accuracy: 0.7281 - val_loss: 2.2356
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 21s 55ms/step - accuracy: 0.7334 - loss: 2.0671 - val_accuracy: 0.7330 - val_loss: 2.1498
Epoch 4/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - accuracy: 0.7397 - loss: 1.9308 - val_accuracy: 0.7389 - val_loss: 2.0756
Epoch 5/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - accuracy: 0.7470 - loss: 1.8009 - val_accuracy: 0.7434 - val_loss: 2.0171

Training GRU with BERT
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 16s 60ms/step - accuracy: 0.7150 - loss: 3.0905 - val_accuracy: 0.7239 - val_loss: 2.3368
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 55ms/step - accuracy: 0.7284 - loss: 2.1951 - val_accuracy: 0.7307 - val_loss: 2.2044
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 13s 55ms/

In [ ]:
summarize_embedding_results("BERT", bert_results)

,Embedding,Model,Validation Accuracy,Validation Loss
0,BERT,LSTM,0.743416,2.017146
1,BERT,GRU,0.750374,1.926166
2,BERT,SimpleRNN,0.744288,2.005863


In [ ]:
qwen_results = train_all_models_for_one_embedding(
    "Qwen",
    "/content/drive/MyDrive/NLP_embedding/qwen_QA_token_embeddings.npy",
    "/content/drive/MyDrive/NLP_embedding/qwen_answer_token_embeddings.npy",
    epochs=5,
    batch_size=16
)


Training LSTM with Qwen
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 18s 63ms/step - accuracy: 0.7112 - loss: 3.2670 - val_accuracy: 0.7189 - val_loss: 2.4910
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - accuracy: 0.7226 - loss: 2.3805 - val_accuracy: 0.7226 - val_loss: 2.4267
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - accuracy: 0.7256 - loss: 2.2566 - val_accuracy: 0.7242 - val_loss: 2.3751
Epoch 4/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - accuracy: 0.7269 - loss: 2.1412 - val_accuracy: 0.7254 - val_loss: 2.3153
Epoch 5/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - accuracy: 0.7300 - loss: 2.0333 - val_accuracy: 0.7265 - val_loss: 2.2755

Training GRU with Qwen
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 17s 62ms/step - accuracy: 0.7125 - loss: 3.1936 - val_accuracy: 0.7188 - val_loss: 2.4957
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 19s 58ms/step - accuracy: 0.7224 - loss: 2.3911 - val_accuracy: 0.7225 - val_loss: 2.4335
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/

In [ ]:
summarize_embedding_results("Qwen", qwen_results)

,Embedding,Model,Validation Accuracy,Validation Loss
0,Qwen,LSTM,0.726474,2.275457
1,Qwen,GRU,0.727412,2.284711
2,Qwen,SimpleRNN,0.721112,2.535193


In [ ]:
e5_results = train_all_models_for_one_embedding(
    "E5",
    "/content/drive/MyDrive/NLP_embedding/e5_QA_token_embeddings.npy",
    "/content/drive/MyDrive/NLP_embedding/e5_answer_token_embeddings.npy",
    epochs=5,
    batch_size=16
)


Training LSTM with E5
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 22s 63ms/step - accuracy: 0.7151 - loss: 3.2954 - val_accuracy: 0.7203 - val_loss: 2.4431
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 19s 78ms/step - accuracy: 0.7238 - loss: 2.3493 - val_accuracy: 0.7233 - val_loss: 2.3899
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - accuracy: 0.7254 - loss: 2.2637 - val_accuracy: 0.7249 - val_loss: 2.3339
Epoch 4/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - accuracy: 0.7272 - loss: 2.1801 - val_accuracy: 0.7277 - val_loss: 2.2736
Epoch 5/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - accuracy: 0.7290 - loss: 2.1004 - val_accuracy: 0.7273 - val_loss: 2.2410

Training GRU with E5
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 17s 63ms/step - accuracy: 0.7162 - loss: 3.1378 - val_accuracy: 0.7222 - val_loss: 2.4300
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - accuracy: 0.7242 - loss: 2.3452 - val_accuracy: 0.7231 - val_loss: 2.3870
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step

In [ ]:
summarize_embedding_results("E5", e5_results)

,Embedding,Model,Validation Accuracy,Validation Loss
0,E5,LSTM,0.727280,2.240984
1,E5,GRU,0.724944,2.311014
2,E5,SimpleRNN,0.715914,2.549067


In [ ]:
bge_results = train_all_models_for_one_embedding(
    "BGE",
    "/content/drive/MyDrive/NLP_embedding/bge_QA_token_embeddings.npy",
    "/content/drive/MyDrive/NLP_embedding/bge_answer_token_embeddings.npy",
    epochs=5,
    batch_size=16
)


Training LSTM with BGE
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 18s 64ms/step - accuracy: 0.7159 - loss: 3.1799 - val_accuracy: 0.7222 - val_loss: 2.3936
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - accuracy: 0.7250 - loss: 2.2722 - val_accuracy: 0.7254 - val_loss: 2.3030
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - accuracy: 0.7286 - loss: 2.1364 - val_accuracy: 0.7273 - val_loss: 2.2170
Epoch 4/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - accuracy: 0.7320 - loss: 2.0015 - val_accuracy: 0.7303 - val_loss: 2.1458
Epoch 5/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 20s 58ms/step - accuracy: 0.7354 - loss: 1.8751 - val_accuracy: 0.7335 - val_loss: 2.0993

Training GRU with BGE
Epoch 1/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 17s 63ms/step - accuracy: 0.7162 - loss: 3.1114 - val_accuracy: 0.7228 - val_loss: 2.3840
Epoch 2/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - accuracy: 0.7251 - loss: 2.2752 - val_accuracy: 0.7253 - val_loss: 2.2902
Epoch 3/5
241/241 ━━━━━━━━━━━━━━━━━━━━ 20s 57ms/st

In [ ]:
summarize_embedding_results("BGE", bge_results)

,Embedding,Model,Validation Accuracy,Validation Loss
0,BGE,LSTM,0.733514,2.099288
1,BGE,GRU,0.736508,2.031103
2,BGE,SimpleRNN,0.729780,2.157561


In [ ]:
summary = pd.DataFrame([
    ["FastText", "LSTM", 0.775837, 1.921146],
    ["FastText", "GRU", 0.807599, 1.641071],
    ["FastText", "SimpleRNN", 0.788321, 1.790974],

    ["BERT", "LSTM", 0.743416, 2.017146],
    ["BERT", "GRU", 0.750374, 1.926166],
    ["BERT", "SimpleRNN", 0.744288, 2.005863],

    ["Qwen", "LSTM", 0.726474, 2.275457],
    ["Qwen", "GRU", 0.727412, 2.284711],
    ["Qwen", "SimpleRNN", 0.721112, 2.535193],

    ["E5", "LSTM", 0.727280, 2.240984],
    ["E5", "GRU", 0.724944, 2.310114],
    ["E5", "SimpleRNN", 0.715914, 2.549067],

    ["BGE", "LSTM", 0.733514, 2.099288],
    ["BGE", "GRU", 0.736508, 2.031103],
    ["BGE", "SimpleRNN", 0.729780, 2.157561],
], columns=["Embedding", "Model", "Validation Accuracy", "Validation Loss"])

summary
summary.sort_values(by=["Validation Accuracy", "Validation Loss"],ascending=[False, True])

,Embedding,Model,Validation Accuracy,Validation Loss
1,FastText,GRU,0.807599,1.641071
2,FastText,SimpleRNN,0.788321,1.790974
0,FastText,LSTM,0.775837,1.921146
4,BERT,GRU,0.750374,1.926166
5,BERT,SimpleRNN,0.744288,2.005863
3,BERT,LSTM,0.743416,2.017146
13,BGE,GRU,0.736508,2.031103
12,BGE,LSTM,0.733514,2.099288
14,BGE,SimpleRNN,0.729780,2.157561
7,Qwen,GRU,0.727412,2.284711


In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def ids_to_text(ids):
    words = []
    for idx in ids:
        if idx == 0:
            continue

        word = target_tokenizer.index_word.get(int(idx), "")

        if word == "endtoken":
            break

        if word != "starttoken":
            words.append(word)

    return " ".join(words)

In [ ]:
def calculate_bleu_score(model, X_test, dec_in_test, dec_tar_test):
    preds = model.predict([X_test, dec_in_test], verbose=0)

    pred_ids = np.argmax(preds, axis=-1)
    true_ids = dec_tar_test.squeeze(-1)

    predictions = []
    references = []

    for p, t in zip(pred_ids, true_ids):
        predictions.append(ids_to_text(p))
        references.append(ids_to_text(t))

    smooth = SmoothingFunction().method1

    references_tok = [[r.split()] for r in references]
    predictions_tok = [p.split() for p in predictions]

    bleu_score = corpus_bleu(
        references_tok,
        predictions_tok,
        smoothing_function=smooth
    )

    return bleu_score

In [ ]:
bleu_fasttext_gru = calculate_bleu_score(
    fasttext_results["GRU"]["model"],
    fasttext_results["GRU"]["X_test"],
    fasttext_results["GRU"]["dec_in_test"],
    fasttext_results["GRU"]["dec_tar_test"]
)

print("BLEU Score:", bleu_fasttext_gru)

BLEU Score: 0.04475908519099116


In [ ]:
preds = fasttext_results["GRU"]["model"].predict(
    [
        fasttext_results["GRU"]["X_test"],
        fasttext_results["GRU"]["dec_in_test"]
    ],
    verbose=0
)

pred_ids = np.argmax(preds, axis=-1)

for i in range(5):

    print("="*50)

    print("Reference:")
    print(ids_to_text(
        fasttext_results["GRU"]["dec_tar_test"][i].squeeze()
    ))

    print()

    print("Prediction:")
    print(ids_to_text(pred_ids[i]))

    print()

Reference:
الخطوات الاساسية تشمل تحليل المتطلبات، تصميم الواجهة، البرمجة، الاختبار، ونشر التطبيق

Prediction:
الاجابة الاساسية تشمل تحليل البيانات استخدام

Reference:
يري البعض ان وسايل التواصل الاجتماعي تساهم في تعزيز التواصل، بينما يعتقد اخرون انها تودي الي تدهور العلاقات الاجتماعية بسبب قلة التواصل الشخصي

Prediction:
اذا الشخص ان وسايل استخدام التعليم تحسين في تحسين استخدام بينما بينما ان لانها تودي الي تعزيز التعليم الاقتصادية بسبب فرص استخدام

Reference:
يعتقد البعض ان الابتكار هو المحرك الرييسي للنمو الاقتصادي لانه يودي الي تحسين الكفاة، وتطوير منتجات جديدة، وزيادة التنافسية في السوق

Prediction:
يعتبر يعتبر ان التعليم هو هو البييي الطاقة الاقتصادي لانه يودي الي تحسين وتعزيز تقنيات جديدة وزيادة في الاردن

Reference:
اري ان التكنولوجيا تلعب دورا حيويا في تسهيل حياتنا اليومية وزيادة الانتاجية، لكنها قد تودي ايضا الي تقليل التفاعل البشري المباشر

Prediction:
ان ان التعليم تكمن دورا مهم في تحسين الطاقة الطبيعية وزيادة الطاقة بينما قد يودي لانه الي تقليل استخدام الطاقة

Reference:
ال

# Transformer based models

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

In [ ]:
input_texts = ["answer the question: " + q for q in questions]
target_texts = answers

In [ ]:
train_inputs, val_inputs, train_targets, val_targets = train_test_split(input_texts,target_texts,test_size=0.2,random_state=42)

In [ ]:
train_dataset = Dataset.from_dict({
    "input_text": train_inputs,
    "target_text": train_targets
})

val_dataset = Dataset.from_dict({
    "input_text": val_inputs,
    "target_text": val_targets
})